In [26]:
import json
import re
import os

def extract_last_number(text: str):
    if text is None:
        return None
    nums = re.findall(r"-?\d+(?:\.\d+)?", text)
    return nums[-1] if nums else None


def load_jsonl(path, max_lines=None):
    data = []
    with open(path, "r", encoding="utf-8") as f:
        for i, line in enumerate(f):
            if max_lines is not None and i >= max_lines:
                break
            data.append(json.loads(line))
    return data


def compute_pass_rate(gt_data, resp_path, max_samples):
    if not os.path.exists(resp_path):
        return None

    resp_data = load_jsonl(resp_path, max_samples)
    total = min(len(gt_data), len(resp_data))

    correct = 0
    for gt, resp in zip(gt_data[:total], resp_data[:total]):
        assert gt['prompt'] == resp['prompt']
        gold = extract_last_number(gt.get("chosen", ""))
        pred = extract_last_number(resp.get("response", ""))

        if gold is None or pred is None:
            continue
        if gold == pred:
            correct += 1

    return correct / total if total > 0 else 0.0


def eval_gsm8k(
    model_name: str,
    gt_path: str = "../src/data/gsm8k/prob_test_gen.jsonl",
    exp_root: str = "../src/exp_results",
    max_epoch: int | None = None,
    max_samples: int = 388,
):
    # -------- load GT once
    gt_data = load_jsonl(gt_path, max_samples)

    results = {}
    ep = 1

    while True:
        if max_epoch is not None and ep > max_epoch:
            break

        resp_path = os.path.join(
            exp_root,
            model_name,
            f"{model_name}_ep{ep}",
            "prob_test_gen_response.jsonl",
        )

        acc = compute_pass_rate(gt_data, resp_path, max_samples)
        results[f"ep{ep}"] = acc
        ep += 1

    return results

In [27]:
model = "eval_llama32_1b_dpo_base_gsm8k_ep6"
eval_gsm8k(model,max_epoch=6)

{'ep1': 0.07731958762886598,
 'ep2': 0.04896907216494845,
 'ep3': 0.05412371134020619,
 'ep4': 0.04639175257731959,
 'ep5': 0.06701030927835051,
 'ep6': 0.061855670103092786}

In [28]:
model = "eval_llama32_1b_dpo_extend_gsm8k_ep6"
eval_gsm8k(model,max_epoch=6)

{'ep1': 0.07989690721649484,
 'ep2': 0.08247422680412371,
 'ep3': 0.06701030927835051,
 'ep4': 0.07731958762886598,
 'ep5': 0.06701030927835051,
 'ep6': 0.05670103092783505}